# RAG Assignment: Baseline vs. Re-Ranking

**Document:** FIDE Laws of Chess (`LawsOfChess.pdf`)

**Pipelines compared over the same chunk corpus:**
1. **Baseline** – Bi-Encoder retrieves Top 10 by cosine similarity, Top 3 go to the LLM → *Answer 1*
2. **Re-Ranking** – Cross-Encoder re-scores those same 10, new Top 3 go to the LLM → *Answer 2*

## 1. Setup & Environment

| Component | Choice | Why |
|---|---|---|
| LLM | `llama3` (8B) via local **Ollama** | Free, runs offline |
| Bi-Encoder | `sentence-transformers/all-MiniLM-L6-v2` | Standard, fast dense embedding model |
| Cross-Encoder | `cross-encoder/ms-marco-MiniLM-L-6-v2` | Suggested in the assignment; trained for passage re-ranking |
| Vector store | **FAISS** (`IndexFlatIP` on normalised vectors = cosine similarity) | Simple, exact search |
| PDF parsing | `pypdf` | Lightweight text extraction |

In [18]:
# Install dependencies into the kernel's own Python environment
import sys
print("Kernel Python:", sys.executable)
%pip install -q pypdf sentence-transformers faiss-cpu ollama numpy pandas

# Ollama must be running locally with the LLM pulled:
#   ollama pull llama3

Kernel Python: /opt/homebrew/opt/python@3.10/bin/python3.10


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # avoids a kernel crash on macOS when torch and faiss are used together

import re
import time

import numpy as np
import pandas as pd
import ollama
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

# ---- Configuration ----
PDF_PATH = "LawsOfChess.pdf"
LLM_MODEL = "llama3"
BI_ENCODER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

TOP_K_RETRIEVE = 10  # candidates from the bi-encoder
TOP_K_CONTEXT = 3    # chunks passed to the LLM

### Check that the local Ollama LLM responds

In [20]:
available = [m.model for m in ollama.list().models]
print("Ollama models:", available)

reply = ollama.chat(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is castling in chess?"}],
    options={"temperature": 0},
)
print(reply.message.content)

Ollama models: ['nomic-embed-text:latest', 'llama3:latest']
Castling is a special move in chess that allows the king to move three squares towards an adjacent rook, while the rook moves to the square the king crossed, in order to get the king to a safer position behind a wall of pawns.


### Load the PDF
Extract the raw text page by page so we can inspect it before chunking.

In [21]:
reader = PdfReader(PDF_PATH)
pages = [page.extract_text() or "" for page in reader.pages]

print(f"Pages: {len(pages)}")
print(f"Total characters: {sum(len(p) for p in pages):,}")
print("\n--- Sample (page 3) ---\n")
print(pages[2][:800])

Pages: 25
Total characters: 54,075

--- Sample (page 3) ---

 3 
Article 2: The initial position of the pieces on the chessboard 
 
2.1 The chessboard is composed of an 8 x 8 grid of 64 equal squares alternately light (the 
‘white’ squares) and dark (the ‘black’ squares). 
The chessboard is placed between the players in such a way that the near corner square 
to the right of the player is white. 
 
2.2 At the beginning of the game one player has 16 light -coloured pieces (the ‘white’ pieces); 
the other has 16 dark-coloured pieces (the ‘black’ pieces). 
 
These pieces are as follows: 
A white king usually indicated by the symbol 
  
A white queen usually indicated by the symbol 
  
Two white rooks usually indicated by the symbol 
  
Two white bishops usually indicated by the symbol 
  
Two white knights usually indicated by the symbol 
  
Eight whit


## 2. Chunking

The Laws of Chess are written as numbered clauses (`3.8`, `9.2`, `A.4` …), and each clause is one rule. So we make **one chunk per clause**. Very long clauses are cut into pieces of at most 150 words.

In [22]:
# Join all pages (skip page 1 = table of contents) and remove the page number at the top of each page
text = "\n".join(re.sub(r"^\s*\d+\s*\n", "", page) for page in pages[1:])

# Split wherever a new clause number like "3.8" or "A.4" starts a line
clauses = re.split(r"\n\s*(?=(?:\d{1,2}|[A-F])\.\d{1,2}\s)", text)

MAX_WORDS = 150
chunks = []
for clause in clauses:
    words = clause.split()
    for i in range(0, len(words), MAX_WORDS):
        chunks.append(" ".join(words[i:i + MAX_WORDS]))

chunks_df = pd.DataFrame({"chunk_id": range(len(chunks)), "text": chunks})
print("Number of chunks:", len(chunks_df))
chunks_df.head(10)

Number of chunks: 132


,chunk_id,text
0,0,FIDE Laws of Chess cover over-the-board play. ...
1,1,special factors. FIDE appeals to all chess pla...
2,2,1.1 The game of chess is played between two op...
3,3,1.2 The objective of each player is to place t...
4,4,1.3 If the position is such that neither playe...
5,5,2.1 The chessboard is composed of an 8 x 8 gri...
6,6,2.2 At the beginning of the game one player ha...
7,7,2.3 The initial position of the pieces on the ...
8,8,2.4 The eight vertical columns of squares are ...
9,9,3.1 It is not permitted to move a piece to a s...


## 3. Indexing with the Bi-Encoder (FAISS)

Each chunk is embedded **independently** by the bi-encoder. We normalise the vectors, so the inner product in `faiss.IndexFlatIP` is the same as **cosine similarity**.

In [23]:
bi_encoder = SentenceTransformer(BI_ENCODER_MODEL)

chunk_embeddings = bi_encoder.encode(chunks, normalize_embeddings=True)

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print("Embedding shape:", chunk_embeddings.shape)
print("Vectors in index:", index.ntotal)

/opt/homebrew/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding shape: (132, 384)
Vectors in index: 132


## 4. Two-Stage Retrieval

### Pipeline 1 – Baseline retrieval (Bi-Encoder)
Embed the query with the same bi-encoder and get the **Top 10** chunks by cosine similarity from FAISS.

In [24]:
def retrieve(query, k=TOP_K_RETRIEVE):
    query_embedding = bi_encoder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(query_embedding, k)
    return pd.DataFrame({"chunk_id": ids[0], "bi_score": scores[0], "bi_rank": range(1, k + 1)})

### Pipeline 2 – Re-ranking (Cross-Encoder)
Pass all 10 `(query, chunk)` pairs through the cross-encoder, then sort by its score (highest first).

In [25]:
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)

def rerank(query, candidates):
    pairs = [(query, chunks[i]) for i in candidates["chunk_id"]]
    reranked = candidates.copy()
    reranked["ce_score"] = cross_encoder.predict(pairs)
    reranked = reranked.sort_values("ce_score", ascending=False)
    reranked["ce_rank"] = range(1, len(reranked) + 1)
    return reranked

### LLM answer generation
Both pipelines use the **same prompt**. Only the 3 context chunks are different.

In [26]:
PROMPT = """You are answering questions about the FIDE Laws of Chess.
Answer the question using ONLY the context below.
If the answer is not in the context, say "Not found in the context."

Context:
{context}

Question: {question}
Answer:"""

def generate(query, chunk_ids):
    context = "\n\n".join(chunks[i] for i in chunk_ids)
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=query)}],
        options={"temperature": 0},
    )
    return response.message.content

### Compare both pipelines for one query
Runs both pipelines, times each stage, and prints how the ranks changed plus Answer 1 vs Answer 2.

In [27]:
def compare(query):
    # Pipeline 1: bi-encoder top 10 -> top 3 -> LLM
    t0 = time.perf_counter()
    candidates = retrieve(query)
    t_retrieve = time.perf_counter() - t0

    t0 = time.perf_counter()
    answer1 = generate(query, candidates["chunk_id"][:TOP_K_CONTEXT])
    t_llm1 = time.perf_counter() - t0

    # Pipeline 2: same top 10 -> cross-encoder -> new top 3 -> LLM
    t0 = time.perf_counter()
    reranked = rerank(query, candidates)
    t_rerank = time.perf_counter() - t0

    t0 = time.perf_counter()
    answer2 = generate(query, reranked["chunk_id"][:TOP_K_CONTEXT])
    t_llm2 = time.perf_counter() - t0

    # Rank table
    table = reranked[["chunk_id", "bi_rank", "ce_rank", "bi_score", "ce_score"]].copy()
    table["text"] = [chunks[i][:80] for i in table["chunk_id"]]
    print("QUERY:", query, "\n")
    print(table.to_string(index=False))
    print("\nAnswer 1 (baseline):\n", answer1)
    print("\nAnswer 2 (re-ranked):\n", answer2)

    return {"query": query, "retrieve_s": t_retrieve, "rerank_s": t_rerank,
            "llm1_s": t_llm1, "llm2_s": t_llm2}

## 5. Failure Mode Analysis (Stress Testing)

**Approach**
1. Write many candidate queries, each tagged with the chunk(s) that **actually** answer it (found by reading the document) and a failure type.
2. For every query, run only retrieval + re-ranking (fast, no LLM) and record the rank of the correct chunk under the bi-encoder and under the cross-encoder.
3. Keep the queries where the **baseline fails** (correct chunk outside the Top 3) but **re-ranking fixes it** (correct chunk moved into the Top 3).

We tried around 160 queries in total. For most of them the bi-encoder already ranked the correct chunk in the Top 3, because many questions share words with the rule that answers them. The list below is a representative subset: every query that passed the filter, plus a mix that did not.

In [28]:
# (query, correct chunk ids, failure type)
candidate_queries = [
    # Keyword trap: the query shares words with other passages, the answer is somewhere unexpected
    ("In blind chess, how is castling announced?", [116], "Keyword trap"),
    ("What symbol is used to mark a check?", [113], "Keyword trap"),
    ("What names are used for the files in blind chess?", [116], "Keyword trap"),
    ("What happens when a player runs out of time?", [40], "Keyword trap"),
    ("What happens if a player shows up late to the game?", [36], "Keyword trap"),
    ("Can I go to the bathroom during my game?", [74], "Keyword trap"),
    ("How is a pawn promotion written on the scoresheet?", [112], "Keyword trap"),
    ("How many points does a player get for a draw?", [72], "Keyword trap"),

    # Exceptions: a general rule exists, but the answer is a special case stated elsewhere
    ("In Chess960, does the king always move when castling?", [125], "Exception"),
    ("Do the special touch rules for blind players also apply to the sighted player?", [117], "Exception"),
    ("What happens if the starting position of the pieces was wrong in rapidplay?", [94], "Exception"),
    ("I ran out of time but my opponent only has a bare king. Do I lose?", [40], "Exception"),
    ("Is castling still possible if the king moved and came back to its square?", [17], "Exception"),
    ("How many illegal moves can a player make before losing the game?", [49], "Exception"),
    ("If I touch my rook first and then my king, can I still castle?", [23], "Exception"),

    # Negation: "not allowed", "forbidden", "cannot"
    ("Can a player not move a piece again once he released it?", [25], "Negation"),
    ("When is castling not allowed?", [17, 18], "Negation"),
    ("Is a player not allowed to castle after his king was in check earlier?", [17, 18], "Negation"),
    ("In blitz, can a player with less than two minutes left claim a draw?", [98], "Negation"),
    ("Do I lose if I move a piece with the wrong hand?", [20], "Negation"),
    ("What are players forbidden to do during play?", [75], "Negation"),
    ("Is it forbidden to write the moves down in advance?", [52], "Negation"),

    # Document-specific: the same topic appears in the main Laws AND in an appendix
    # (Rapidplay, Blitz, blind players, Chess960), so the specific rule gets buried
    ("In rapidplay, after how many moves can you no longer complain about piece placement?", [94], "Document-specific"),
    ("In rapidplay, does the arbiter rule on touch-move on his own?", [94], "Document-specific"),
    ("For a blind player, when is a piece considered touched?", [116, 117], "Document-specific"),
    ("In Chess960, can castling happen on the very first move?", [125], "Document-specific"),
    ("What if the player who sealed the move is absent at resumption?", [129], "Document-specific"),
    ("What is the penalty for making an illegal move?", [49], "Document-specific"),
    ("What happens after a player makes an illegal move in blitz?", [98], "Document-specific"),
    ("How is a win on time claimed in rapidplay?", [95], "Document-specific"),
]
print("Candidate queries:", len(candidate_queries))

Candidate queries: 30


In [29]:
def rank_of_correct(ranked_ids, correct_ids):
    """1-based position of the first correct chunk in the list, or None if it is not there."""
    for pos, chunk_id in enumerate(ranked_ids, start=1):
        if chunk_id in correct_ids:
            return pos
    return None

rows = []
for query, correct_ids, failure_type in candidate_queries:
    candidates = retrieve(query)
    reranked = rerank(query, candidates)
    rows.append({
        "type": failure_type,
        "query": query,
        "correct_chunk": correct_ids,
        "bi_rank": rank_of_correct(candidates["chunk_id"].tolist(), correct_ids),
        "ce_rank": rank_of_correct(reranked["chunk_id"].tolist(), correct_ids),
    })

stress_df = pd.DataFrame(rows)
stress_df   # bi_rank / ce_rank = NaN means the correct chunk was not in the Top 10

,type,query,correct_chunk,bi_rank,ce_rank
0,Keyword trap,"In blind chess, how is castling announced?",[116],6.0,1.0
1,Keyword trap,What symbol is used to mark a check?,[113],5.0,1.0
2,Keyword trap,What names are used for the files in blind chess?,[116],8.0,3.0
3,Keyword trap,What happens when a player runs out of time?,[40],5.0,4.0
4,Keyword trap,What happens if a player shows up late to the ...,[36],2.0,4.0
5,Keyword trap,Can I go to the bathroom during my game?,[74],1.0,4.0
6,Keyword trap,How is a pawn promotion written on the scoresh...,[112],3.0,1.0
7,Keyword trap,How many points does a player get for a draw?,[72],1.0,1.0
8,Exception,"In Chess960, does the king always move when ca...",[125],5.0,2.0
9,Exception,Do the special touch rules for blind players a...,[117],7.0,3.0


In [30]:
# Baseline fails (correct chunk not in its Top 3) but re-ranking puts it in the Top 3
failures = stress_df[(stress_df["bi_rank"].isna() | (stress_df["bi_rank"] > 3)) & (stress_df["ce_rank"] <= 3)]
print(f"{len(failures)} of {len(stress_df)} queries: baseline fails, re-ranking fixes it")
failures

10 of 30 queries: baseline fails, re-ranking fixes it


,type,query,correct_chunk,bi_rank,ce_rank
0,Keyword trap,"In blind chess, how is castling announced?",[116],6.0,1.0
1,Keyword trap,What symbol is used to mark a check?,[113],5.0,1.0
2,Keyword trap,What names are used for the files in blind chess?,[116],8.0,3.0
8,Exception,"In Chess960, does the king always move when ca...",[125],5.0,2.0
9,Exception,Do the special touch rules for blind players a...,[117],7.0,3.0
10,Exception,What happens if the starting position of the p...,[94],7.0,2.0
15,Negation,Can a player not move a piece again once he re...,[25],6.0,2.0
22,Document-specific,"In rapidplay, after how many moves can you no ...",[94],7.0,1.0
23,Document-specific,"In rapidplay, does the arbiter rule on touch-m...",[94],5.0,2.0
24,Document-specific,"For a blind player, when is a piece considered...","[116, 117]",4.0,1.0


In [31]:
# The other direction: re-ranking pushed the correct chunk OUT of the Top 3, or it was never in the Top 10
worse = stress_df[(stress_df["bi_rank"] <= 3) & (stress_df["ce_rank"] > 3)]
missed = stress_df[stress_df["bi_rank"].isna()]
print("Re-ranking made it worse:")
print(worse[["query", "bi_rank", "ce_rank"]].to_string(index=False))
print("\nCorrect chunk not in the bi-encoder's Top 10 (re-ranking cannot help):")
print(missed[["query", "correct_chunk"]].to_string(index=False))

Re-ranking made it worse:
                                              query  bi_rank  ce_rank
What happens if a player shows up late to the game?      2.0      4.0
           Can I go to the bathroom during my game?      1.0      4.0
   Do I lose if I move a piece with the wrong hand?      2.0      6.0

Correct chunk not in the bi-encoder's Top 10 (re-ranking cannot help):
                                     query correct_chunk
How is a win on time claimed in rapidplay?          [95]


### Selected failure cases: Answer 1 vs Answer 2

- **Part A (≥2 failure queries):** the two biggest rank jumps.
- **Part B (document-specific failures):** one query per failure type.
- **Limitation:** one query where the correct chunk never reaches the Top 10, so re-ranking cannot fix it.

In [32]:
selected_queries = [
    # Part A: failure queries where the baseline fails and re-ranking fixes it
    "In rapidplay, after how many moves can you no longer complain about piece placement?",  # #7 -> #1
    "For a blind player, when is a piece considered touched?",                               # #4 -> #1
    # Part B: one per failure type
    "In blind chess, how is castling announced?",                      # Keyword trap        #6 -> #1
    "In Chess960, does the king always move when castling?",           # Exception           #5 -> #2
    "Can a player not move a piece again once he released it?",        # Negation            #6 -> #2
    "In rapidplay, does the arbiter rule on touch-move on his own?",   # Document-specific   #5 -> #2
    # Limitation: correct chunk never reaches the Top 10, so re-ranking cannot help
    "How is a win on time claimed in rapidplay?",
]

timings = []
for q in selected_queries:
    timings.append(compare(q))
    print("\n" + "=" * 120 + "\n")


QUERY: In rapidplay, after how many moves can you no longer complain about piece placement? 

 chunk_id  bi_rank  ce_rank  bi_score  ce_score                                                                             text
       94        7        1  0.499283  3.774165 A.4 Where supervision is inadequate the Competition Rules shall apply, except wh
       91        1        2  0.588509 -0.209566 A.1 A ‘Rapidplay’ game is one where either all the moves must be made in a fixed
       24        9        3  0.492813 -3.183094 4.5 If none of the pieces touched can be moved or captured, the player may make 
       69        3        4  0.532443 -4.054922 10.1 A ‘quickplay finish’ is the phase of a game when all the ( remaining) moves
       48        5        5  0.502891 -4.850533 7.3 If a player displaces one or more pieces, he shall re -establish the correct
       21        8        6  0.494803 -6.252930 4.2 Provided that he first expresses his intention ( for example by saying „j’ad
   

## 6. Latency Profiling

Timings were recorded inside `compare()` with `time.perf_counter()` for the 7 selected queries.
- **Baseline pipeline** = bi-encoder retrieval + LLM
- **Re-ranking pipeline** = bi-encoder retrieval + cross-encoder re-ranking + LLM

In [ ]:
latency_df = pd.DataFrame(timings)
latency_df["baseline_retrieval_s"] = latency_df["retrieve_s"]
latency_df["reranking_retrieval_s"] = latency_df["retrieve_s"] + latency_df["rerank_s"]
latency_df["baseline_total_s"] = latency_df["retrieve_s"] + latency_df["llm1_s"]
latency_df["reranking_total_s"] = latency_df["retrieve_s"] + latency_df["rerank_s"] + latency_df["llm2_s"]

latency_df.loc["Average"] = latency_df.mean(numeric_only=True)
latency_df.round(3)

In [ ]:
avg = latency_df.loc["Average"]
summary = pd.DataFrame({
    "Baseline (ms)": [avg["retrieve_s"] * 1000, 0, avg["llm1_s"] * 1000, avg["baseline_total_s"] * 1000],
    "Re-ranking (ms)": [avg["retrieve_s"] * 1000, avg["rerank_s"] * 1000, avg["llm2_s"] * 1000, avg["reranking_total_s"] * 1000],
}, index=["Bi-encoder retrieval (Top 10)", "Cross-encoder re-ranking (10 pairs)", "LLM generation", "Total"])
summary.round(1)